# Revela — Notebook 04: Inference Pipeline Requirements

**Issues:** F1.1 (#49), F1.3 (#51), F1.5 (#53), D3.7 (#123)  
**Model:** Revela dermoscopic CNN — EfficientNet-B0  

### Purpose
Define requirements for the Revela inference pipeline. The pipeline takes a dermoscopic image as input and returns a structured classification result from the CNN. It does **not** generate natural-language reasoning — that is handled separately by static templates (H1.1–H1.4).

### What Inference Produces
```
Input:  Dermoscopic image (JPEG/PNG)
Output: {
    model_version: str,
    top_class: str,
    top_confidence: float,
    top3_predictions: [{class, confidence, confidence_pct}],
    safety_note: str,
    reasoning_template_id: str
}
```

> Revela is a CNN classifier. It outputs class labels and confidence scores. It does not provide clinical reasoning, diagnosis, or treatment advice.

## Block 1 — Model Registry Schema (F1.1 — #49)

The inference pipeline must be config-driven. No model-specific logic should be hard-coded.

In [ ]:
model_registry_schema = """
# config/model_registry.yaml

models:

  dermoscopic_baseline_v1:
    input_type: dermoscopic
    architecture: efficientnet_b0
    num_classes: 3
    class_names:
      - Melanoma
      - Benign nevus
      - Other lesion
    checkpoint_path: models/bcn20000_effnet_b0/best_model.pth
    class_to_idx_path: models/bcn20000_effnet_b0/class_to_idx.json
    image_size: 224
    status: baseline_only  # Other lesion mixes malignant + benign — not a valid cancer-risk classifier

  dermoscopic_cancer_risk_v2:
    input_type: dermoscopic
    architecture: efficientnet_b0
    num_classes: 4
    class_names:
      - Melanoma
      - Non-melanoma skin cancer
      - Benign nevus
      - Other non-cancer / indeterminate lesion
    checkpoint_path: models/bcn20000_cancer_risk_effnet_b0/best_model.pth
    class_to_idx_path: models/bcn20000_cancer_risk_effnet_b0/class_to_idx.json
    image_size: 224
    status: planned  # Depends on D3.3 (#118) and D3.4 (#119)

  clinical_skin_condition_v1:
    input_type: clinical_photo
    architecture: efficientnet_b0
    num_classes: 5
    class_names:
      - Eczema / dermatitis
      - Urticaria / allergic reaction
      - Folliculitis / acne-like
      - Psoriasis / papulosquamous
      - Lesion — dermoscopic review recommended
    checkpoint_path: models/clinical_skin_condition_effnet_b0/best_model.pth
    class_to_idx_path: models/clinical_skin_condition_effnet_b0/class_to_idx.json
    image_size: 224
    status: planned  # Depends on V2.5 (#125)
"""

print(model_registry_schema)

## Block 2 — Model Loader Requirements (F1.1 — #49)

**File:** `src/inference/model_loader.py`

| Requirement | Detail |
|-------------|--------|
| Config-driven | Load model metadata from `model_registry.yaml` by `model_id` |
| No hard-coded classes | Class names must come from registry, not source code |
| Device handling | Auto-detect: CUDA → MPS → CPU |
| Class mapping | Load from `class_to_idx.json` or equivalent |
| Extensible | Adding a new model requires only a new registry entry, not code changes |
| Smoke test | CLI or unit test that loads `dermoscopic_baseline_v1` checkpoint without error |

**Supported model IDs at MVP:**
- `dermoscopic_baseline_v1` — smoke testing only (3-class, old taxonomy)
- `dermoscopic_cancer_risk_v2` — production dermoscopic CNN (4-class, cancer-risk)
- `clinical_skin_condition_v1` — clinical photo CNN (5-class, future)

## Block 3 — Preprocessing Requirements

**File:** `src/inference/preprocessor.py`

| Step | Spec |
|------|------|
| Resize | 224 × 224 px (from registry `image_size`) |
| Colour | RGB |
| Normalise | ImageNet mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]` |
| Format | `torch.Tensor` shape `[1, 3, 224, 224]` (batch dim included) |
| Input formats | JPEG, PNG |
| Rejected inputs | Non-image files, corrupt files → raise `ValueError` with message |

In [2]:
preprocessing_spec = """
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_inference_transform(image_size: int = 224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
"""

print('=== PREPROCESSING SPEC ===')
print(preprocessing_spec)

=== PREPROCESSING SPEC ===

from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_inference_transform(image_size: int = 224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])



## Block 4 — Inference Function Requirements

**File:** `src/inference/predict.py`

| Requirement | Detail |
|-------------|--------|
| Input | PIL Image or file path |
| Forward pass | `model.eval()`, `torch.no_grad()` |
| Output format | Softmax probabilities (not raw logits) |
| Top-3 output | Return top 3 predictions sorted by confidence descending (F1.3) |
| Confidence | Float `[0.0, 1.0]` and formatted percentage string |
| Class names | From loaded registry — NOT hard-coded |
| Error handling | File not found, corrupt image, wrong input type → structured error response |

## Block 5 — Top-3 Prediction Output (F1.3 — #51)

The `predict_image()` function must return a `predictions` list with exactly 3 entries.

In [ ]:
example_top3_output = [
    {'rank': 1, 'class': 'Melanoma',                                  'confidence': 0.731, 'confidence_pct': '73.1%'},
    {'rank': 2, 'class': 'Non-melanoma skin cancer',                  'confidence': 0.183, 'confidence_pct': '18.3%'},
    {'rank': 3, 'class': 'Other non-cancer / indeterminate lesion',   'confidence': 0.071, 'confidence_pct': '7.1%'},
]

print('=== EXAMPLE TOP-3 PREDICTION OUTPUT ===')
print(f'{"Rank":<6} {"Class":<45} {"Confidence":<12} {"Pct"}')
print('-' * 75)
for p in example_top3_output:
    print(f'{p["rank"]:<6} {p["class"]:<45} {p["confidence"]:<12.3f} {p["confidence_pct"]}')

print()
print('Rules:')
print('  - Always return exactly 3 predictions')
print('  - Sorted by confidence descending')
print('  - confidence: float [0.0, 1.0]')
print('  - confidence_pct: formatted string with 1 decimal place')
print('  - Sum of all 4 class confidences = 1.0 (softmax)')

## Block 6 — Inference Response Schema (F1.5 — #53)

The full response object returned to the frontend.

In [ ]:
import json

success_response = {
    'status': 'ok',
    'model_version': 'dermoscopic_cancer_risk_v2',
    'top_class': 'Melanoma',
    'top_confidence': 0.731,
    'top3_predictions': [
        {'rank': 1, 'class': 'Melanoma',                                'confidence': 0.731, 'confidence_pct': '73.1%'},
        {'rank': 2, 'class': 'Non-melanoma skin cancer',                'confidence': 0.183, 'confidence_pct': '18.3%'},
        {'rank': 3, 'class': 'Other non-cancer / indeterminate lesion', 'confidence': 0.071, 'confidence_pct': '7.1%'},
    ],
    'safety_note': 'This result is for educational purposes only. It is not a clinical diagnosis. Consult a dermatologist for any skin concern.',
    'reasoning_template_id': 'melanoma',
}

error_response = {
    'status': 'error',
    'error_code': 'INVALID_IMAGE',
    'message': 'Uploaded file could not be processed. Please upload a valid JPEG or PNG image.',
}

print('=== SUCCESS RESPONSE SCHEMA ===')
print(json.dumps(success_response, indent=2))
print()
print('=== ERROR RESPONSE SCHEMA ===')
print(json.dumps(error_response, indent=2))

print()
print('=== FIELD DEFINITIONS ===')
fields = {
    'status':                 'ok | error',
    'model_version':          'Registry model_id used for this inference',
    'top_class':              'Class name of highest-confidence prediction',
    'top_confidence':         'Float [0.0, 1.0] — confidence of top prediction',
    'top3_predictions':       'List of top 3 {rank, class, confidence, confidence_pct}',
    'safety_note':            'Fixed educational-only disclaimer string',
    'reasoning_template_id':  'Slug used to look up static reasoning template (e.g. melanoma, benign_nevus)',
    'error_code':             '(error only) Machine-readable error type',
    'message':                '(error only) Human-readable error description',
}
for field, desc in fields.items():
    print(f'  {field:30s}: {desc}')

## Block 7 — Error Codes

| Error Code | Trigger | HTTP Status |
|------------|---------|-------------|
| `INVALID_IMAGE` | File is not a valid JPEG or PNG | 400 |
| `IMAGE_TOO_LARGE` | File exceeds upload size limit | 413 |
| `MODEL_NOT_LOADED` | Requested model_id not in registry or checkpoint missing | 503 |
| `INFERENCE_FAILED` | Unexpected error during model forward pass | 500 |

All errors must return the `error_response` schema (Block 6). No raw Python tracebacks to frontend.

## Block 8 — Reasoning Template IDs (H1.1 — #62)

The `reasoning_template_id` in the response schema maps to a static template for display. Templates are written separately (H1.1–H1.4) and must not contain diagnostic claims.

| CNN Class | reasoning_template_id | Template Content (summary) |
|-----------|----------------------|---------------------------|
| Melanoma | `melanoma` | Visual features of melanoma, ABCDE rule, dermoscopy note |
| Non-melanoma skin cancer | `nmsc` | BCC/SCC features, pearly borders, ulceration note |
| Benign nevus | `benign_nevus` | Regular structure, symmetry, typical nevus features |
| Other non-cancer / indeterminate lesion | `other` | Variable features, possible pre-cancer note, review advised |

> Templates must not contain treatment advice or diagnostic certainty. All templates include an educational-only disclaimer.

## Block 9 — Summary

| Component | File | Issue |
|-----------|------|-------|
| Model registry config | `config/model_registry.yaml` | F1.1 (#49) |
| Model loader | `src/inference/model_loader.py` | F1.1 (#49) |
| Preprocessor | `src/inference/preprocessor.py` | F1.2 |
| Predict function | `src/inference/predict.py` | F1.2 |
| Top-3 output | Inside `predict.py` | F1.3 (#51) |
| Response schema | Defined in Block 6 | F1.5 (#53) |
| Reasoning template IDs | `src/templates/reasoning_templates.py` | H1.1 (#62) |

### Models Registered
| Model ID | Classes | Status |
|----------|---------|--------|
| `dermoscopic_baseline_v1` | 3 (old taxonomy) | Smoke test only |
| `dermoscopic_cancer_risk_v2` | 4 (cancer-risk) | Planned — depends on #118, #119 |
| `clinical_skin_condition_v1` | 5 (clinical) | Planned — depends on #125 |

### Next Steps
1. Implement `config/model_registry.yaml` (F1.1 — #49 — assigned Bilal)
2. Implement `src/inference/model_loader.py` (F1.1 — #49)
3. Implement `src/inference/preprocessor.py` and `predict.py` (F1.2)
4. Wire top-3 output (F1.3 — #51)
5. Wire response schema to frontend (F1.5 — #53)
6. Update schema for cancer-risk model (D3.7 — #123) once #119 is done